# Session 3: a network in a computer
**A. Real-world problem.** Real networks are too big and too fast to watch. So we build a
toy one: routers are points on a graph, links have a delay and a bandwidth, and packets move
hop by hop through queues. Nothing here touches a real network: no sockets, only arithmetic.
The point is to check our simulator against the formulas of Session 2, and then to break
the network on purpose (full buffers, a cut link, a routing loop) and watch what happens.

**Plan.** (1) Dijkstra's shortest-path algorithm builds routing tables, (2) an event-driven
simulator forwards packets through finite queues, (3) measured delay is compared with
propagation + transmission + queueing, (4) loss and retransmission: stop-and-wait and
window protocols, (5) a link failure and re-routing, (6) failure cases, (7) a simplified
additive-increase multiplicative-decrease (AIMD) sawtooth, (8) a video link mini-project.

In [ ]:
import heapq
import itertools
import math
import pathlib
import random
from collections import Counter, deque

import numpy as np

OUT = (pathlib.Path(__file__).resolve().parent if "__file__" in globals() else pathlib.Path(".")) / "out"

## Step 1: Dijkstra builds the routing tables
`graph[u][v]` is the cost of the link from u to v. Dijkstra keeps a priority queue of the
cheapest known way to reach each node and always expands the cheapest one next. Each router
then keeps only the *next hop* for every destination.

In [ ]:
def dijkstra(graph, source):
    """Return (dist, prev): cheapest cost to every node and the node before it on that path."""
    dist = {source: 0.0}
    prev = {}
    heap = [(0.0, source)]
    done = set()
    while heap:
        d, u = heapq.heappop(heap)
        if u in done:
            continue
        done.add(u)
        for v, cost in graph.get(u, {}).items():
            nd = d + cost
            if v not in dist or nd < dist[v]:
                dist[v] = nd
                prev[v] = u
                heapq.heappush(heap, (nd, v))
    return dist, prev


def routing_tables(graph):
    """{router: {destination: next hop}} from Dijkstra run at every router."""
    tables = {}
    for src in graph:
        _, prev = dijkstra(graph, src)
        row = {}
        for dst in prev:
            hop = dst
            while prev[hop] != src:
                hop = prev[hop]
            row[dst] = hop
        tables[src] = row
    return tables

## Step 2: the simulator
Time jumps from event to event (a packet arrives, a transmission ends). Each directed link has
a *transmitter* that sends one packet at a time (`bits / rate` seconds), a *queue* of waiting
packets of finite length `buffer` (a packet that finds it full is dropped), and a *wire delay*.
Every packet is counted: sent = delivered + dropped + in flight.

In [ ]:
class Packet:
    __slots__ = ("pid", "src", "dst", "bits", "t_sent", "ttl", "kind", "seq")

    def __init__(self, pid, src, dst, bits, t_sent, ttl, kind, seq):
        self.pid, self.src, self.dst, self.bits = pid, src, dst, bits
        self.t_sent, self.ttl, self.kind, self.seq = t_sent, ttl, kind, seq


class Link:
    def __init__(self, u, v, delay, rate, buffer, loss):
        self.u, self.v, self.delay, self.rate, self.buffer, self.loss = u, v, delay, rate, buffer, loss
        self.up = True
        self.queue = deque()
        self.current = None          # the packet being transmitted
        self.carried = 0             # packets that finished transmission on this link


class Network:
    def __init__(self, seed=0):
        self.links = {}
        self.tables = {}
        self.now = 0.0
        self.events = []
        self.counter = itertools.count()
        self.pids = itertools.count()
        self.rng = random.Random(seed)
        self.sent = 0
        self.delivered = []          # (packet, arrival time)
        self.dropped = Counter()     # reason -> number of packets
        self.live = set()            # ids of packets that exist and are not yet delivered or dropped
        self.on_deliver = {}         # node -> function(packet), used by the protocols below

    # ---- building the network
    def add_link(self, u, v, delay, rate, buffer=50, loss=0.0):
        """Add a two-way link: delay in seconds, rate in bit/s, buffer = packets that may wait."""
        self.links[(u, v)] = Link(u, v, delay, rate, buffer, loss)
        self.links[(v, u)] = Link(v, u, delay, rate, buffer, 0.0)

    def cost_graph(self, ref_bits=12000):
        """Link costs for Dijkstra: the delay a packet of `ref_bits` would see on an empty link."""
        g = {}
        for (u, v), lk in self.links.items():
            g.setdefault(u, {})
            g.setdefault(v, {})
            if lk.up:
                g[u][v] = lk.delay + ref_bits / lk.rate
        return g

    def recompute_routes(self):
        self.tables = routing_tables(self.cost_graph())

    # ---- the clock
    def schedule(self, t, fn, *args):
        heapq.heappush(self.events, (t, next(self.counter), fn, args))

    def run(self, until=math.inf):
        while self.events and self.events[0][0] <= until:
            t, _, fn, args = heapq.heappop(self.events)
            self.now = t
            fn(*args)
        if until != math.inf:
            self.now = until

    # ---- packets
    def inject(self, t, src, dst, bits, ttl=64, kind="data", seq=0):
        """Create a packet at time t at node src."""
        self.schedule(t, self.originate, src, dst, bits, ttl, kind, seq)

    def originate(self, src, dst, bits, ttl=64, kind="data", seq=0):
        pkt = Packet(next(self.pids), src, dst, bits, self.now, ttl, kind, seq)
        self.sent += 1
        self.live.add(pkt.pid)
        self.arrive(pkt, src)

    def drop(self, pkt, reason):
        self.dropped[reason] += 1
        self.live.discard(pkt.pid)

    def arrive(self, pkt, node):
        if node == pkt.dst:
            self.delivered.append((pkt, self.now))
            self.live.discard(pkt.pid)
            if node in self.on_deliver:
                self.on_deliver[node](pkt)
            return
        if pkt.ttl <= 0:
            return self.drop(pkt, "ttl")
        nxt = self.tables.get(node, {}).get(pkt.dst)
        if nxt is None:
            return self.drop(pkt, "no_route")
        link = self.links.get((node, nxt))
        if link is None or not link.up:
            return self.drop(pkt, "link_down")
        pkt.ttl -= 1
        if link.current is None:
            self.start_tx(link, pkt)
        elif len(link.queue) >= link.buffer:
            self.drop(pkt, "buffer")
        else:
            link.queue.append(pkt)

    def start_tx(self, link, pkt):
        link.current = pkt
        self.schedule(self.now + pkt.bits / link.rate, self.tx_done, link)

    def tx_done(self, link):
        pkt, link.current = link.current, None
        link.carried += 1
        if link.queue:
            self.start_tx(link, link.queue.popleft())
        if not link.up:
            self.drop(pkt, "link_down")
        elif link.loss > 0 and self.rng.random() < link.loss:
            self.drop(pkt, "loss")
        else:
            self.schedule(self.now + link.delay, self.arrive, pkt, link.v)

    # ---- failures
    def fail_link(self, t, u, v, detect_after):
        """At time t the link u-v breaks; routers learn of it (tables recomputed) detect_after seconds later."""
        self.schedule(t, self._break, u, v)
        self.schedule(t + detect_after, self.recompute_routes)

    def _break(self, u, v):
        for key in ((u, v), (v, u)):
            lk = self.links[key]
            lk.up = False
            while lk.queue:
                self.drop(lk.queue.popleft(), "link_down")

    # ---- bookkeeping
    def in_flight_count(self):
        """Count packets directly: in transmitters, in queues and travelling on wires."""
        n = sum(len(lk.queue) + (1 if lk.current is not None else 0) for lk in self.links.values())
        n += sum(1 for ev in self.events if ev[2] == self.arrive)
        return n

    def totals(self):
        return {"sent": self.sent, "delivered": len(self.delivered), "dropped": sum(self.dropped.values()),
                "in_flight": self.in_flight_count()}

    def conserved(self):
        t = self.totals()
        return t["sent"] == t["delivered"] + t["dropped"] + t["in_flight"] and len(self.live) == t["in_flight"]

    def delays(self, kind="data"):
        return [t - p.t_sent for p, t in self.delivered if p.kind == kind]


def path_delay(net, path, bits):
    """Analytic delay of one packet on an empty path: sum of transmission + propagation on each link."""
    return sum(bits / net.links[(a, b)].rate + net.links[(a, b)].delay for a, b in zip(path, path[1:]))


def burst_last_delay(net, path, bits, n):
    """Analytic delay of the last of n packets sent at the same instant (store and forward)."""
    x = [bits / net.links[(a, b)].rate for a, b in zip(path, path[1:])]
    return sum(x) + (n - 1) * max(x) + sum(net.links[(a, b)].delay for a, b in zip(path, path[1:]))


def line_network(rates, delays, buffer=1000):
    """Nodes n0 - n1 - ... in a line with the given per-link rates (bit/s) and delays (s)."""
    net = Network()
    for i, (r, d) in enumerate(zip(rates, delays)):
        net.add_link(f"n{i}", f"n{i + 1}", d, r, buffer)
    net.recompute_routes()
    return net

## Step 3: loss and retransmission (stop-and-wait and windows)
A sender sends `window` packets and waits for acknowledgements (ACKs). If an ACK does not
come back before the *timeout*, the packet is sent again. Window 1 is *stop-and-wait*.

Define `x = L / R` (time to send one packet) and `RTT` = the time from the end of sending a
packet to the arrival of its ACK = 2 * delay + ACK transmission time. Stop-and-wait sends one
packet per `x + RTT`, so the fraction of time the link is busy is

    U = x / (x + RTT)          and for a window W:   U = min(1, W * x / (x + RTT))

In [ ]:
class Transfer:
    """Send n packets from src to dst with a sliding window and per-packet timeouts."""

    def __init__(self, net, src, dst, n, data_bits, window, timeout, ack_bits=400):
        self.net, self.src, self.dst, self.n = net, src, dst, n
        self.data_bits, self.window, self.timeout, self.ack_bits = data_bits, window, timeout, ack_bits
        self.attempt = [0] * n
        self.next_new = 0
        self.outstanding, self.acked, self.received = set(), set(), set()
        self.transmissions = 0
        self.done_time = None
        net.on_deliver[dst] = self._at_receiver
        net.on_deliver[src] = self._at_sender

    def start(self, t=0.0):
        self.net.schedule(t, self._fill)

    def _send(self, seq):
        self.attempt[seq] += 1
        self.transmissions += 1
        self.net.originate(self.src, self.dst, self.data_bits, 64, "data", seq)
        self.net.schedule(self.net.now + self.timeout, self._timeout, seq, self.attempt[seq])

    def _fill(self):
        while self.next_new < self.n and len(self.outstanding) < self.window:
            seq = self.next_new
            self.next_new += 1
            self.outstanding.add(seq)
            self._send(seq)

    def _timeout(self, seq, attempt):
        if seq in self.outstanding and self.attempt[seq] == attempt:
            self._send(seq)

    def _at_receiver(self, pkt):
        if pkt.kind == "data":
            self.received.add(pkt.seq)
            self.net.originate(self.dst, self.src, self.ack_bits, 64, "ack", pkt.seq)

    def _at_sender(self, pkt):
        if pkt.kind == "ack" and pkt.seq in self.outstanding:
            self.outstanding.discard(pkt.seq)
            self.acked.add(pkt.seq)
            if len(self.acked) == self.n:
                self.done_time = self.net.now
            else:
                self._fill()


def two_node_network(rate=10e6, delay=0.010, buffer=1000, loss=0.0, seed=0):
    net = Network(seed)
    net.add_link("S", "D", delay, rate, buffer, loss)
    net.recompute_routes()
    return net


def utilisation(n, data_bits, window, rate=10e6, delay=0.010, ack_bits=400, loss=0.0, timeout=1.0, seed=0):
    """Run a transfer and return (measured U, transfer, network). U = useful sending time / total time."""
    net = two_node_network(rate, delay, loss=loss, seed=seed)
    tr = Transfer(net, "S", "D", n, data_bits, window, timeout, ack_bits)
    tr.start()
    net.run()
    return n * data_bits / rate / tr.done_time, tr, net


def utilisation_formula(data_bits, window, rate=10e6, delay=0.010, ack_bits=400):
    x = data_bits / rate
    rtt = 2 * delay + ack_bits / rate
    return min(1.0, window * x / (x + rtt))

## Step 4: a toy congestion window (a simplified illustration)
Real TCP is far more complex. In this toy, once per round trip the sender sends `w` packets.
The path can deliver `bdp` packets per round trip (the bandwidth-delay product) and a buffer
holds `buffer` more; packets beyond `bdp + buffer` are lost. No loss: `w` grows by 1 (additive
increase). Loss: `w` is halved (multiplicative decrease). The graph of `w` is a sawtooth.

In [ ]:
def aimd(bdp, buffer, rounds, w0=1.0):
    """Return per-round arrays: window, delivered and dropped packets of the toy AIMD model."""
    w = float(w0)
    window, delivered, dropped = [], [], []
    capacity = bdp + buffer
    for _ in range(rounds):
        window.append(w)
        lost = max(0.0, w - capacity)
        delivered.append(w - lost)
        dropped.append(lost)
        w = max(1.0, w / 2) if lost > 0 else w + 1.0
    return np.array(window), np.array(delivered), np.array(dropped)


def aimd_utilisation(bdp, buffer, rounds=2000):
    """Fraction of the path's capacity (bdp packets per round trip) used, after the start-up."""
    w, _, _ = aimd(bdp, buffer, rounds)
    used = np.minimum(w, bdp)
    return float(used[rounds // 2:].mean() / bdp)

## Step 5: failure cases
* **Routing loop:** two routers each think the other is the way to the destination. The
  packet ping-pongs until its *time to live* (TTL) runs out. Without TTL it would circle forever.
* **Missing route:** a router has no row for the destination and must drop the packet.

In [ ]:
def has_loop(tables, src, dst):
    seen = {src}
    node = src
    while node != dst:
        node = tables.get(node, {}).get(dst)
        if node is None:
            return False
        if node in seen:
            return True
        seen.add(node)
    return False


def diamond():
    """A - R1 - R2 - B (short, 5 ms links) and A - R3 - R4 - B (long, 10 ms links), all 10 Mbit/s."""
    net = Network()
    for a, b, d in (("A", "R1", 0.005), ("R1", "R2", 0.005), ("R2", "B", 0.005),
                    ("A", "R3", 0.010), ("R3", "R4", 0.010), ("R4", "B", 0.010)):
        net.add_link(a, b, d, 10e6, buffer=100)
    net.recompute_routes()
    return net


def reroute_experiment(fail_at=0.100, detect_after=0.050, interval=0.004, offset=0.001, end=0.4, bits=12000):
    """A steady flow A -> B; the link R1-R2 breaks at `fail_at`. Returns (network, send times)."""
    net = diamond()
    times = [offset + k * interval for k in range(int((end - offset) / interval))]
    for t in times:
        net.inject(t, "A", "B", bits)
    net.fail_link(fail_at, "R1", "R2", detect_after)
    net.run()
    return net, times

## Step 6: mini-project, a video stream over one link
A camera sends 25 frames per second. Each frame is 20000 bytes of payload that the sender cuts
into packets (payload `P` bytes plus a 40 byte header) and hands to the link all at once. The
link has rate 5 Mbit/s and 10 ms delay, and its router has a buffer of `B` waiting packets.

In [ ]:
VIDEO = {"frame_bytes": 20000, "header": 40, "fps": 25, "rate": 5e6, "delay": 0.010, "frames": 60}


def video_trial(payload, buffer):
    """Simulate the stream; return (frames fully delivered, worst frame delay in s, packets dropped)."""
    v = VIDEO
    n = math.ceil(v["frame_bytes"] / payload)
    bits = (payload + v["header"]) * 8
    net = Network()
    net.add_link("cam", "viewer", v["delay"], v["rate"], buffer)
    net.recompute_routes()
    for f in range(v["frames"]):
        for _ in range(n):
            net.inject(f / v["fps"], "cam", "viewer", bits, seq=f)
    net.run()
    got = Counter(p.seq for p, _ in net.delivered)
    last = {}
    for p, t in net.delivered:
        last[p.seq] = max(last.get(p.seq, 0.0), t - p.t_sent)
    complete = [f for f in range(v["frames"]) if got[f] == n]
    worst = max((last[f] for f in complete), default=float("inf"))
    return len(complete), worst, sum(net.dropped.values())


def min_buffer(payload, limit=60):
    """Smallest buffer (packets) that loses no packet of the stream, found by trying 0, 1, 2, ..."""
    for b in range(limit + 1):
        if video_trial(payload, b)[2] == 0:
            return b
    return None


def smallest_payload(buffer, lo=761, hi=1460):
    """Smallest payload in [lo, hi] bytes that loses nothing with `buffer` waiting packets (bisection:
    a bigger payload never needs more packets per frame). Returns None if even `hi` loses packets."""
    if video_trial(hi, buffer)[2] > 0:
        return None
    while lo < hi:
        mid = (lo + hi) // 2
        if video_trial(mid, buffer)[2] == 0:
            hi = mid
        else:
            lo = mid + 1
    return lo

In [ ]:
if __name__ == "__main__":
    import matplotlib.pyplot as plt

    OUT.mkdir(exist_ok=True)
    # --- routing
    g = {"A": {"B": 4, "C": 1}, "B": {"A": 4, "C": 2, "D": 1}, "C": {"A": 1, "B": 2, "D": 5}, "D": {"B": 1, "C": 5}}
    dist, _ = dijkstra(g, "A")
    print("Dijkstra from A:", dist, " next hops:", routing_tables(g)["A"])

    # --- delay validation: one packet over a 3-link path with different rates
    net = line_network([10e6, 2e6, 10e6], [0.005, 0.020, 0.005])
    net.inject(0.0, "n0", "n3", 12000)
    net.run()
    sim = net.delays()[0]
    ana = path_delay(net, ["n0", "n1", "n2", "n3"], 12000)
    print(f"one packet: simulated {1000 * sim:.3f} ms, analytic {1000 * ana:.3f} ms")

    # --- burst through one link: queueing grows 1.2 ms per packet, buffer of 4 drops the rest
    net = two_node_network(buffer=4)
    for _ in range(10):
        net.inject(0.0, "S", "D", 12000)
    net.run()
    d = np.array(sorted(net.delays()))
    print("burst of 10, buffer 4: delivered", len(d), "dropped", dict(net.dropped), "delays (ms)", np.round(1000 * d, 2))

    # --- stop-and-wait and windows
    for w in (1, 4, 8, 18, 64):
        u, tr, _ = utilisation(2000, 12000, w)
        print(f"window {w:2d}: measured U = {u:.4f}, formula {utilisation_formula(12000, w):.4f}")
    u, tr, net = utilisation(3000, 12000, 1, loss=0.2, timeout=0.05, seed=1)
    print("stop-and-wait with 20 percent loss: all delivered", len(tr.acked) == 3000, ", transmissions", tr.transmissions,
          ", U", round(u, 4))

    # --- link failure
    net, times = reroute_experiment()
    t = net.totals()
    print("reroute:", t, dict(net.dropped))
    for s in (0.021, 0.081, 0.201, 0.301):
        pk = [(p, tt) for p, tt in net.delivered if abs(p.t_sent - s) < 1e-9]
        print(f"  packet sent at {s:.3f} s: delay {1000 * (pk[0][1] - s):.2f} ms" if pk else f"  packet sent at {s:.3f} s: lost")
    fig, ax = plt.subplots(figsize=(6.5, 4))
    got = {round(p.t_sent, 6): tt - p.t_sent for p, tt in net.delivered}
    ax.plot([s for s in times if round(s, 6) in got], [1000 * got[round(s, 6)] for s in times if round(s, 6) in got], "o", ms=3, label="delivered")
    lost = [s for s in times if round(s, 6) not in got]
    ax.plot(lost, [0] * len(lost), "rx", label="lost")
    ax.axvline(0.100, color="gray", ls=":")
    ax.axvline(0.150, color="gray", ls=":")
    ax.set_xlabel("time the packet was sent (s)")
    ax.set_ylabel("end-to-end delay (ms)")
    ax.set_title("Link cut at 0.10 s, routes updated at 0.15 s")
    ax.grid(alpha=0.3)
    ax.legend()
    fig.tight_layout()
    fig.savefig(OUT / "reroute.png", dpi=120)

    # --- delay validation figure
    fig2, axs = plt.subplots(1, 2, figsize=(10, 4))
    k = np.arange(len(d))
    axs[0].plot(k + 1, 1000 * d, "o", label="simulated (delivered)")
    axs[0].plot(k + 1, 1000 * ((k + 1) * 0.0012 + 0.010), "-", label="(k x 1.2 ms) + 10 ms formula")
    axs[0].set_xlabel("packet number in the burst (k)")
    axs[0].set_ylabel("end-to-end delay (ms)")
    axs[0].set_title("Burst of 10, buffer of 4: 5 delivered, 5 dropped")
    axs[0].grid(alpha=0.3)
    axs[0].legend()
    ws = np.arange(1, 25)
    meas = [utilisation(600, 12000, int(w))[0] for w in ws]
    axs[1].plot(ws, meas, "o", label="simulated")
    axs[1].plot(ws, [utilisation_formula(12000, int(w)) for w in ws], "-", label="min(1, W x / (x + RTT))")
    axs[1].set_xlabel("window W (packets)")
    axs[1].set_ylabel("link utilisation U")
    axs[1].set_title("Window protocol: simulation against the formula")
    axs[1].grid(alpha=0.3)
    axs[1].legend()
    fig2.tight_layout()
    fig2.savefig(OUT / "delay_and_window.png", dpi=120)

    # --- AIMD
    w, dl, dr = aimd(bdp=20, buffer=20, rounds=200)
    print("AIMD bdp 20, buffer 20: peak", w[100:].max(), "min", w[100:].min(), "mean", round(w[100:].mean(), 2),
          "sent", w.sum(), "delivered", dl.sum(), "dropped", dr.sum())
    for buf in (0, 10, 20):
        print(f"  buffer {buf:2d}: utilisation {aimd_utilisation(20, buf):.3f}")
    fig3, ax3 = plt.subplots(figsize=(6.5, 4))
    ax3.plot(w, label="window w")
    ax3.axhline(40, color="red", ls="--", label="pipe + buffer = 40")
    ax3.axhline(20, color="gray", ls=":", label="pipe (bdp) = 20")
    ax3.set_xlabel("round trip number")
    ax3.set_ylabel("window (packets)")
    ax3.set_title("Toy AIMD sawtooth (simplified illustration)")
    ax3.grid(alpha=0.3)
    ax3.legend()
    fig3.tight_layout()
    fig3.savefig(OUT / "aimd_sawtooth.png", dpi=120)

    # --- failure cases
    net = diamond()
    net.tables["R2"]["B"] = "R1"
    print("loop:", has_loop(net.tables, "A", "B"))
    net.inject(0.0, "A", "B", 12000, ttl=10)
    net.run()
    print("   traversals", sum(lk.carried for lk in net.links.values()), "dropped", dict(net.dropped))
    net = diamond()
    del net.tables["R1"]["B"]
    net.inject(0.0, "A", "B", 12000)
    net.run()
    print("missing route:", dict(net.dropped))

    # --- mini-project
    print("payload  packets/frame  min buffer  worst frame delay (ms)  overhead")
    for payload in (400, 761, 800, 1000, 1200, 1460):
        n = math.ceil(20000 / payload)
        b = min_buffer(payload)
        done, worst, lost = video_trial(payload, b)
        print(f"{payload:6d}  {n:12d}  {b:10d}  {1000 * worst:20.2f}  {100 * 40 / (40 + payload):.2f}%   memory {b * (payload + 40)} bytes")
    best = smallest_payload(16)
    print("router with 16 buffer slots: smallest loss-free payload", best, "->", video_trial(best, 16),
          "; one byte less:", video_trial(best - 1, 16))

## Where this stops
Real routers use distributed protocols instead of one global Dijkstra, real senders adapt
their window to feedback, and queues are managed actively. The simulator is a model; the
roadmap in `promo/roadmap.md` shows how to take it toward the Target.

## Coding tasks
1. **Easy.** Run `utilisation(2000, 12000, W)` for W = 2, 3 and 5 and compare each result with `utilisation_formula`. Predict the first one by hand.
2. **Stretch.** In `reroute_experiment` set `interval=0.002` and predict how many packets are lost with the same detection time of 0.05 s before you run it.
3. **Explain.** Explain why in the toy AIMD model a buffer of 20 packets keeps the pipe full while a buffer of 0 gives about 75 percent, using the peak and low values of the sawtooth.

## Assessment
- **A. Conceptual.** Why is stop-and-wait very inefficient on a link with a long delay, and what does a window change?
- **B. Numerical.** A sender uses stop-and-wait with 1500 byte packets on a 10 Mbit/s link; the RTT after sending is 20.04 ms. Find x, U and the throughput. Which window fills the pipe?
- **C. Graph and simulation interpretation.** In the re-routing graph, read the delay before the cut, after re-routing, and the number of lost packets. Why are exactly those packets lost, and how many would be lost if the failure were detected after 0.025 s instead of 0.05 s?
- **D. Design and optimisation challenge.** Design: the router is upgraded from 16 to 20 buffer slots. A 20000 byte video frame is sent as one burst; each packet has a 40 byte header. What is the smallest payload that loses nothing now, and what happens with a payload of 761 bytes?

## Checkpoint
Run this last cell. It prints PASS when your work is correct.

In [ ]:
import itertools
import math
import random


def _best_by_brute_force(graph, src, dst):
    """Cheapest cost over all simple paths (only for tiny graphs): a check that does not use Dijkstra."""
    others = [n for n in graph if n not in (src, dst)]
    best = math.inf
    for r in range(len(others) + 1):
        for mid in itertools.permutations(others, r):
            path = [src, *mid, dst]
            if all(b in graph[a] for a, b in zip(path, path[1:])):
                best = min(best, sum(graph[a][b] for a, b in zip(path, path[1:])))
    return best


def run_checks(L):
    """L is the lesson module (or notebook namespace). Returns a list of (label, passed)."""
    checks = []

    # --- Dijkstra
    g = {"A": {"B": 4, "C": 1}, "B": {"A": 4, "C": 2, "D": 1}, "C": {"A": 1, "B": 2, "D": 5}, "D": {"B": 1, "C": 5}}
    dist, _ = L.dijkstra(g, "A")
    checks.append(("hand calculation: from A the costs are B 3, C 1, D 4 (A-C-B-D)",
                   dist == {"A": 0.0, "B": 3.0, "C": 1.0, "D": 4.0}))
    tab = L.routing_tables(g)
    checks.append(("the next hop from A to D is C, and from D to A is B",
                   tab["A"]["D"] == "C" and tab["D"]["A"] == "B"))
    rng = random.Random(4)
    names = list("ABCDEFG")
    big = {n: {} for n in names}
    for a, b in itertools.combinations(names, 2):
        if rng.random() < 0.5:
            w = rng.randint(1, 9)
            big[a][b] = big[b][a] = w
    agree = True
    for a in names:
        d, _ = L.dijkstra(big, a)
        for b in names:
            if a != b:
                agree &= abs(d.get(b, math.inf) - _best_by_brute_force(big, a, b)) < 1e-12
    checks.append(("Dijkstra equals brute-force search over all paths for every pair in a random 7-node graph", agree))
    iso = {"A": {"B": 1}, "B": {"A": 1}, "Z": {}}
    checks.append(("an unreachable node gets no route", "Z" not in L.routing_tables(iso)["A"]))

    # --- delay validation
    net = L.line_network([10e6, 2e6, 10e6], [0.005, 0.020, 0.005])
    net.inject(0.0, "n0", "n3", 12000)
    net.run()
    sim = net.delays()[0]
    checks.append((f"one packet over 3 links: simulated {1000 * sim:.3f} ms equals the sum of transmission and propagation, 38.400 ms",
                   abs(sim - 0.0384) < 1e-12 and abs(L.path_delay(net, ["n0", "n1", "n2", "n3"], 12000) - 0.0384) < 1e-12))
    net = L.line_network([10e6, 2e6, 10e6], [0.005, 0.020, 0.005])
    for _ in range(6):
        net.inject(0.0, "n0", "n3", 12000)
    net.run()
    last = max(net.delays())
    checks.append(("a burst of 6 over links of 10, 2 and 10 Mbit/s: last packet delay = sum of x + 5 * slowest x + delays",
                   abs(last - L.burst_last_delay(net, ["n0", "n1", "n2", "n3"], 12000, 6)) < 1e-12))

    # --- queue, buffer, loss (worked example of Section E)
    net = L.two_node_network(buffer=4)
    for _ in range(10):
        net.inject(0.0, "S", "D", 12000)
    net.run()
    d = sorted(net.delays())
    checks.append((f"worked example: burst of 10, buffer 4: 5 delivered, 5 dropped, last delivered delay 16.0 ms (got {len(d)}, {net.dropped['buffer']}, {1000 * d[-1]:.1f})",
                   len(d) == 5 and net.dropped["buffer"] == 5 and abs(d[-1] - 0.016) < 1e-12))
    checks.append(("delay of packet k in the burst is k * 1.2 ms + 10 ms (k = 1..5), so queueing adds 1.2 ms per packet ahead",
                   all(abs(x - (k * 0.0012 + 0.010)) < 1e-12 for k, x in enumerate(d, 1))))
    checks.append(("queueing delay of the last delivered packet is 4 * 1.2 = 4.8 ms", abs((d[-1] - d[0]) - 0.0048) < 1e-12))
    net0 = L.two_node_network(buffer=0)
    for _ in range(2):
        net0.inject(0.0, "S", "D", 12000)
    net0.run()
    checks.append(("edge case: with no buffer a burst of 2 delivers 1 and drops 1", len(net0.delivered) == 1 and net0.dropped["buffer"] == 1))

    # --- conservation at many stop times
    lossy = L.two_node_network(buffer=5, loss=0.1, seed=7)
    for i in range(60):
        lossy.inject(i * 0.0005, "S", "D", 12000)
    ok, busy = True, 0
    for stop in (0.0, 0.003, 0.0171, 0.02, 0.05, 0.1):
        lossy.run(until=stop)
        ok &= lossy.conserved()
        busy += lossy.totals()["in_flight"] > 0
    lossy.run()
    end = lossy.totals()
    checks.append(("sent = delivered + dropped + in flight at six stop times", ok and busy >= 3))
    checks.append(("at the end nothing is in flight and 60 sent = delivered + dropped",
                   end["in_flight"] == 0 and end["sent"] == 60 and end["delivered"] + end["dropped"] == 60 and lossy.dropped["buffer"] > 0 and lossy.dropped["loss"] > 0))

    # --- stop-and-wait and windows
    x, rtt = 12000 / 10e6, 2 * 0.010 + 400 / 10e6
    u1, tr1, _ = L.utilisation(2000, 12000, 1)
    checks.append((f"stop-and-wait U = x / (x + RTT) = 1.2 / 21.24 = 0.0565 (got {u1:.4f})",
                   abs(u1 - x / (x + rtt)) < 1e-9 and abs(u1 - 0.056497) < 1e-5 and len(tr1.acked) == 2000))
    ok = True
    for w, target in ((4, 0.2260), (8, 0.4520)):
        u, _, _ = L.utilisation(2000, 12000, w)
        ok &= abs(u - L.utilisation_formula(12000, w)) < 0.005 * target and abs(u - target) < 0.001
    checks.append(("windows of 4 and 8 give utilisation 0.226 and 0.452 as W x / (x + RTT) says", ok))
    u18, _, _ = L.utilisation(2000, 12000, 18)
    checks.append(("a window of 18 fills the pipe: utilisation above 0.99, and equals N x / (N x + RTT) for N = 2000",
                   u18 > 0.99 and abs(u18 - 2000 * x / (2000 * x + rtt)) < 1e-4))
    p = 0.2
    ul, trl, netl = L.utilisation(3000, 12000, 1, loss=p, timeout=0.05, seed=1)
    per = (x + rtt) + p / (1 - p) * 0.05
    checks.append((f"with 20 percent loss every packet still arrives; transmissions per packet about 1/(1-p) = 1.25 (got {trl.transmissions / 3000:.3f})",
                   len(trl.acked) == 3000 and len(trl.received) == 3000 and abs(trl.transmissions / 3000 - 1.25) < 0.03))
    checks.append((f"with loss, U = x / ((x + RTT) + p/(1-p) * timeout) = 0.0356 within 5 percent (got {ul:.4f})",
                   abs(ul - x / per) < 0.05 * x / per))
    checks.append(("lossy transfer keeps the packet count balanced", netl.conserved() and netl.totals()["in_flight"] == 0))

    checks.append(("x = 1.2 ms, RTT after sending = 20.04 ms, and a window of 18 is the smallest that fills the pipe",
                   abs(x - 0.0012) < 1e-12 and abs(rtt - 0.02004) < 1e-12 and math.ceil((x + rtt) / x) == 18
                   and L.utilisation_formula(12000, 17) < 1.0 and L.utilisation_formula(12000, 18) == 1.0))
    checks.append(("stop-and-wait throughput is 0.0565 * 10 Mbit/s = 0.565 Mbit/s", abs(u1 * 10 - 0.565) < 0.001))

    # --- link failure and re-routing
    net, times = L.reroute_experiment()
    arrivals = [0.001 + 0.004 * k + 0.005 + 12000 / 10e6 for k in range(len(times))]
    expected_lost = sum(1 for a in arrivals if 0.100 - 0.0012 <= a < 0.150)
    t = net.totals()
    checks.append((f"link R1-R2 cut at 0.10 s, routes updated at 0.15 s: {expected_lost} packets lost, as counted from arrival times at R1 (got {net.dropped['link_down']})",
                   net.dropped["link_down"] == expected_lost == 13 and t["sent"] == 99 and t["delivered"] == 86))
    byt = {round(p.t_sent, 6): tt - p.t_sent for p, tt in net.delivered}
    before = [v for s, v in byt.items() if s < 0.09]
    after = [v for s, v in byt.items() if s > 0.2]
    checks.append(("before the cut every delay is 18.6 ms (3 links of 5 + 1.2 ms); after re-routing it is 33.6 ms (3 links of 10 + 1.2 ms)",
                   all(abs(v - 0.0186) < 1e-9 for v in before) and all(abs(v - 0.0336) < 1e-9 for v in after) and len(before) > 10 and len(after) > 10))
    checks.append(("the re-routed path is the shortest path of the remaining graph", L.diamond().tables["A"]["B"] == "R1"
                   and net.tables["A"]["B"] == "R3"))
    checks.append(("packet count balanced through the failure", net.conserved() and t["in_flight"] == 0))

    net2, times2 = L.reroute_experiment(detect_after=0.025)
    arr2 = [0.001 + 0.004 * k + 0.005 + 0.0012 for k in range(len(times2))]
    checks.append(("detecting the failure after 0.025 s instead of 0.05 s loses 7 packets instead of 13",
                   net2.dropped["link_down"] == 7 == sum(1 for a in arr2 if 0.1 - 0.0012 <= a < 0.125)))

    # --- failure cases
    net = L.diamond()
    net.tables["R2"]["B"] = "R1"
    net.inject(0.0, "A", "B", 12000, ttl=10)
    net.run()
    hops = sum(lk.carried for lk in net.links.values())
    checks.append(("routing loop: detected by has_loop, the packet makes exactly 10 hops (its TTL) and is dropped, never delivered",
                   L.has_loop(net.tables, "A", "B") and hops == 10 and net.dropped["ttl"] == 1 and not net.delivered and net.conserved()))
    net = L.diamond()
    checks.append(("the healthy tables contain no loop", not L.has_loop(net.tables, "A", "B")))
    del net.tables["R1"]["B"]
    net.inject(0.0, "A", "B", 12000)
    net.run()
    checks.append(("missing route: the packet is dropped at R1 with reason no_route", net.dropped["no_route"] == 1 and not net.delivered))

    # --- AIMD toy model
    w, dl, dr = L.aimd(20, 20, 400)
    tail = w[200:]
    checks.append((f"AIMD sawtooth: window climbs to about 40 then halves to about 20 (peak {tail.max():.1f}, low {tail.min():.1f})",
                   abs(tail.max() - 40) < 0.5 and abs(tail.min() - 20) < 0.5))
    checks.append((f"AIMD mean window is about 3/4 of the peak (got {tail.mean() / tail.max():.3f})", abs(tail.mean() / tail.max() - 0.75) < 0.02))
    checks.append(("AIMD packets: sent = delivered + dropped", abs(w.sum() - dl.sum() - dr.sum()) < 1e-9))
    u0, u10, u20 = (L.aimd_utilisation(20, b) for b in (0, 10, 20))
    checks.append((f"buffer 0, 10 and 20 give utilisation about 0.75, 0.95 and 1.00 (got {u0:.3f}, {u10:.3f}, {u20:.3f})",
                   abs(u0 - 0.75) < 0.01 and abs(u10 - 0.954) < 0.01 and u20 > 0.999))

    # --- mini-project
    n = math.ceil(20000 / 1177)
    done, worst, lost = L.video_trial(1177, 16)
    checks.append((f"mini-project: payload 1177 gives 17 packets per frame, loses nothing with 16 buffer slots, worst frame delay 43.10 ms (got {n}, {lost}, {1000 * worst:.2f})",
                   n == 17 and lost == 0 and done == 60 and abs(worst - (17 * 1217 * 8 / 5e6 + 0.010)) < 1e-9))
    done2, _, lost2 = L.video_trial(1176, 16)
    checks.append(("payload 1176 needs 18 packets per frame and loses one per frame with 16 slots, so no frame is complete",
                   math.ceil(20000 / 1176) == 18 and lost2 == 60 and done2 == 0))
    checks.append(("the smallest loss-free payload for 16 slots is 1177, found by bisection", L.smallest_payload(16) == 1177))
    checks.append(("overhead of that choice is 40 / 1217 = 3.29 percent, below 5 percent", 40 / 1217 < 0.05 and abs(40 / 1217 - 0.0329) < 5e-5))
    checks.append(("the smallest loss-free buffer is n - 1: 13 slots for payload 1460, 26 for payload 761",
                   L.min_buffer(1460) == 13 and L.min_buffer(761) == 26))
    checks.append(("payload 761 into 20 slots: 27 packets per frame, 6 lost from each of the 60 frames",
                   math.ceil(20000 / 761) == 27 and L.video_trial(761, 20)[2] == 360))
    checks.append(("worst frame delay of the chosen design is below the 45 ms limit", worst < 0.045))
    checks.append(("a router with 20 slots needs payload 953 (21 packets per frame), overhead 40 / 993 = 4.03 percent",
                   L.smallest_payload(20) == 953 and math.ceil(20000 / 953) == 21 and abs(40 / 993 - 0.0403) < 5e-5))
    return checks

import types
_ns = types.SimpleNamespace(**{k: v for k, v in globals().items() if not k.startswith('_')})
_ok = True
for _label, _passed in run_checks(_ns):
    print(('PASS  ' if _passed else 'FAIL  ') + _label)
    _ok &= bool(_passed)
print('PASS' if _ok else 'FAIL')
